# 03 — Parallel vs. Sequential Workflows

*Level 6 — Multi-Agent RAG*

## Objective
Run the same multi-agent task both ways and measure the real wall-clock difference — not assume parallel is faster.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "supervisor", "workflows"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
import time
from multiagent_common.dataset import prepare
from multiagent_common.retrieval import DenseRetriever
from multiagent_common.loader import load_agent_class
from sequential import run_sequential
from parallel import run_parallel

data = prepare()
corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)

RetrievalAgent = load_agent_class("retrieval-agent", "RetrievalAgent")
SqlAgent = load_agent_class("sql-agent", "SqlAgent")
WebAgent = load_agent_class("web-agent", "WebAgent")

agents = {
    "retrieval-agent": RetrievalAgent(retriever, data.corpus),
    "sql-agent": SqlAgent(),
    "web-agent": WebAgent(),
}
task = "Summarize what you know about company debt and how many films are in the database."


In [3]:
start = time.time()
sequential_results = run_sequential(task, agents)
sequential_elapsed = time.time() - start
print(f"Sequential: {sequential_elapsed:.2f}s")


Sequential: 9.48s


In [4]:
parallel_results, parallel_elapsed = run_parallel(task, agents)
print(f"Parallel:   {parallel_elapsed:.2f}s")
print(f"Speedup:    {sequential_elapsed / parallel_elapsed:.2f}x")


Parallel:   5.45s
Speedup:    1.74x


## Confirm both produced the same 3 agents' worth of results


In [5]:
print("Sequential agents:", list(sequential_results.keys()))
print("Parallel agents:  ", list(parallel_results.keys()))
for name in agents:
    print(f"  {name}: sequential success={sequential_results[name].success}, parallel success={parallel_results[name].success}")


Sequential agents: ['retrieval-agent', 'sql-agent', 'web-agent']
Parallel agents:   ['retrieval-agent', 'sql-agent', 'web-agent']
  retrieval-agent: sequential success=True, parallel success=True
  sql-agent: sequential success=False, parallel success=False
  web-agent: sequential success=True, parallel success=True


## What I observed

**Real measured speedup: 1.74x** (9.48s sequential vs. 5.45s parallel) for 3 agents each making their own Ollama/SQLite/web calls — not the naive "3 agents = 3x faster" a reader might assume. Parallel wall-clock time is bounded below by the *slowest* individual agent call, not divided evenly across all of them, so the real speedup depends on how unbalanced the agents' individual latencies are.

**`sql-agent` failed in both runs** — the shared task ("Summarize what you know about company debt *and* how many films are in the database") is a compound question mixing two unrelated asks, and broadcasting that same raw text to every agent gave the SQL agent nothing coherent to turn into one query. This is a real limitation of this simple workflow, not a SQL-agent bug: a production supervisor should decompose a compound task into a tailored sub-task *per agent* before delegating, rather than handing every agent the identical original text.

The trade-off `workflows/sequential.py` exists for: only sequential execution lets a later agent's task be *augmented* with an earlier agent's actual finding (`carry_context=True`) — parallel agents never see each other's output before they finish.

## Next

[Level 7 — Production RAG](../../07-production-rag/README.md) — coordinating agents is one problem; running the whole system reliably, securely, and observably in production is the next.
